In [ ]:
from api.server_dao.analysis import DAOAnalysis
from config import init_all_polish_models

from typing import List
from tqdm import tqdm

from dao.attribute import DAOAttributePL

from models.attribute import AttributePL

from analysis.attribute_retriving import perform_full_analysis
from analysis.nlp_transformations import preprocess_text
from services.utils import suppress_stdout
from datetime import datetime
import hashlib
from api.api_models.analysis import Analysis, AnalysisType, AnalysisStatus

In [ ]:
init_all_polish_models()

In [ ]:


dao_document = DAODocument()
dao_analysis = DAOAnalysis()
dao_attributes = DAOAttributePL(collection_name="attributes", db_name="ananas_prod")
documents: List[DocumentInDB] = dao_document.find_many_by_query({"owner_id": "2"})

In [ ]:
for document in tqdm(documents, total=len(documents),
                                desc=f'Calculating lab reports statistics', unit='Lab reports', miniters=1):
        text_to_analyse = preprocess_text(document.plaintext_content)
        with suppress_stdout():
            analysis_result = perform_full_analysis(text_to_analyse, 'pl')
        attribute_to_insert = AttributePL(
            referenced_db_name='documents',
            referenced_doc_id=document.id,
            language="pl",
            is_generated=None,
            is_personal=None,
            **analysis_result.dict()
        )
        dao_attributes.insert_one(attribute_to_insert)

        current_analysis_id = hashlib.sha256(f"{document.document_hash}_{AnalysisType.DOCUMENT_LEVEL}_2".encode()).hexdigest()

        analysis = Analysis(
            analysis_id=current_analysis_id,
            type=AnalysisType.DOCUMENT_LEVEL,
            status=AnalysisStatus.FINISHED,
            document_hash=document.document_hash,
            estimated_wait_time=0,
            start_time=datetime.now()
        )
        dao_analysis.insert_one(analysis)

        current_analysis_id = hashlib.sha256(f"{document.document_hash}_{AnalysisType.CHUNK_LEVEL}_2".encode()).hexdigest()

        analysis = Analysis(
            analysis_id=current_analysis_id,
            type=AnalysisType.CHUNK_LEVEL,
            status=AnalysisStatus.FINISHED,
            document_hash=document.document_hash,
            estimated_wait_time=0,
            start_time=datetime.now()
        )
        dao_analysis.insert_one(analysis)


In [ ]:
og_text = """
<<Imię>> <<Nazwisko>> (<<nr albumu>>), <<Imię>> <<Nazwisko>> (<<nr albumu>>)
Politechnika Warszawska
Komputerowe i Sieciowe Systemy Operacyjne
Dokumentacja z realizacji projektu semestralnego
26 lipca 2025
11.1. Serwer aplikacyjny . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6
<<obcy język>>
Oświadczenie
Niniejszy dokument to dokumentacja wstępna projektu w ramach przedmiotu KSO. Oświadczamy, że ta
praca, stanowiąca podstawę do uznania osiągnięcia efektów uczenia się z przedmiotu KSO, została wykonana
przez nas samodzielnie.
1. Tematyka
Tematem projektu jest wykonanie od podstaw budowy własnego systemu informatycznego, który zosta-
nie uruchomiony na edukacyjnej platformie sprzętowej. Zaliczenie projektu zespołowego będzie prowadzone
w oparciu o wykonanie dokumentacji projektowej z projektowania, wdrożenia oraz utrzymania systemu infor-
matycznego.
2. Skład zespołu projektowego
W celu efektywnego zarządzania pracą w zespole, członkom zespołu przydzielono następujące funkcje:
• <<Imię>> <<Nazwisko>> – lider,
• <<Imię>> <<Nazwisko>> – inżynier.
Lider będzie odpowiadać za kwestie formalne takie jak dokumentacja projektu. Inżynier na własnym sprzęcie
utworzy i przetestuje zaprojektowaną sieć.
3. Zakres projektu
W ramach projektu należy zrealizować system informatyczny, w którym znajdzie się:
• serwer oferujący usługę aplikacji www,
<<obcy język>>
• router brzegowy z funkcją firewalla,
• stacja kliencka.
4. Produkty w projekcie
Produktami projektu będą:
• maszyna wirtualna z emulatorem sieci GNS3, na której wdrożony zostanie zaprojektowany system,
• dokumentacja końcowa, w której opisane zostaną kolejne etapy realizacji projektu – w tym również napotkane
problemy, ich rozwiązania itp.
5. Ogólny schemat projektowanego systemu
W celu zobrazowania komunikacji pomiędzy komponentami docelowej sieci zespół przygotował diagram połą-
czeń między systemami. Przedstawia on typy danych i wiadomości wymieniane pomiędzy systemami i klientami.
Ruch powinien być szyfrowany end-to-end pomiędzy Internetem i firewallem aplikacyjnym. Firewall aplikacyjny
może blokować np. złośliwe zapytania HTTP lub próby ataku brute force na zaimplementowany mechanizm
uwierzytelniania. Utworzony diagram połączeń między systemami przedstawia rysunek 1.
<<obcy język>>
Rysunek 1: Diagram połączeń między systemami
W celu zapewnienia odpowiedniego bezpieczeństwa sieci usługowej zespół zdecydował się na dodanie routera
brzegowego z funkcją firewalla i NAT na styku z siecią Internet. W sieci klienckiej, w której znajduje się stacja
kliencka, zdecydowano się dodać również router brzegowy z funkcją NAT. Zasugerowaną topologię sieci na
schemacie wysokopoziomowym przedstawia rysunek 2.
Rysunek 2: Sugerowana topologia sieci: schemat wysokopoziomowy
Wprowadzenie routera i firewalla sieciowego umożliwia:
• wprowadzenie dodatkowej warstwy ochrony sieci usługowej,
• zmniejszenie objętości niedozwolonego ruchu trafiającego do sieci wewnętrznej,
• zmniejszenie objętości złośliwego ruchu trafiającego do WAF.
Odróżnienie funkcjonalności firewalla sieciowego od firewalla aplikacyjnego przedstawia rysunek 3.
Rysunek 3: Odróżnienie funkcjonalności firewalla sieciowego od aplikacyjnego
<<obcy język>>
<<obrazek>>
<<obrazek>>
<<obrazek>>
W celu zdefiniowania założeń projektowych zespół przyjął scenariusz, który przedstawia zleceniodawcę oraz
cel realizacji projektu.
Scenariusz: lokalna spółdzielnia mieszkaniowa poprosiła zespół o przygotowanie bezpiecznej aplikacji webowej.
Zadaniem aplikacji jest przedstawienie danych o spółdzielni, pracownikach, stanowiskach i adresach, które są
obejmowane przez spółdzielnię. Z racji, że spółdzielnia nie posiada jeszcze własnej infrastruktury sieciowej, pro-
jekt musi zostać przetestowany na dedykowanej, bezpiecznej, wirtualnej infrastrukturze sieciowej. Infrastruktura
musi obejmować firewall brzegowy (sieciowy) i aplikacyjny. Infrastruktura sieciowa zostanie następnie wdrożona
u klienta. Założeniami projektowymi są:
• priorytetowo – zrealizowanie funkcjonalności wymaganej przez klienta,
• priorytetowo – realizacja zadanego projektu w terminie,
• optymalizacja kosztów, które ostatecznie poniesie klient (np. kupna i utrzymania infrastruktury),
• zrealizowanie projektu w sposób, który umożliwi jego łatwe utrzymanie,
• zrealizowanie projektu w sposób, który umożliwi jego dalsze rozwijanie,
• odpowiednie zabezpieczenie infrastruktury – wprowadzenie firewalla sieciowego i aplikacyjnego,
• stworzenie dokumentacji umożliwiającej zarządzanie, utrzymanie i rozwijanie systemu.
W celu efektywniejszego zarządzania projektem w przypadku wystąpienia problemów w jego realizacji,
przeprowadzono analizę ryzyk możliwych do wystąpienia. Tabelę z analizą ryzyka opisującą zdarzenie, prawdo-
podobieństwo wystąpienia, wpływ na projekt oraz czynności zaradcze, przedstawia rysunek 4.
Rysunek 4: Tabela z analizą ryzyka
W celu zadbania o efektywność i skuteczność procesu realizacji projektu, jego realizację podzielono na fazy:
• faza dokumentacji: mająca na celu stworzenie dokumentacji obrazującej proces realizacji projektu. Cykliczna,
realizowana stopniowo w trakcie trwania projektu. Związana z pozostałym fazami i efektami ich realizacji.
• faza projektowania: mająca na celu zaprojektowanie docelowego systemu.
• faza przygotowań: mająca na celu zebranie wszystkich niezbędnych elementów, które umożliwią zaimple-
mentowanie komponentów a finalnie utworzenie kompletnego systemu informatycznego.
• faza implementacji: mająca na celu zaimplementowanie zaprojektowanych komponentów i wdrożenie doce-
lowych usług. Obejmuje wdrożenie ich w:
⋄ etapie nieprodukcyjnym: maszyny wirtualne działające w trybie standalone (samodzielnym),
⋄ etapie produkcyjnym: maszyny wirtualne włączone w docelową topologię sieci z innymi współpracującymi
komponentami (sieciowymi, usługowymi).
• faza testowania: mająca na celu weryfikacje zgodności przygotowanych maszyn wirtualnych z założeniami.
Ta faza będzie obejmować:
⋄ testy funkcjonalne: zbadanie oferowanych usług i zależności poszczególnych usług między sobą.
<<obcy język>>
<<obrazek>>
⋄ testy niefunkcjonalne: zbadanie intuicyjności zaprojektowanego systemu informatycznego – w szczegól-
ności usługi oferowanej przez serwer aplikacyjny (testy ze stacji klienckiej). Podstawową grupą badawczą
będą współlokatorzy twórców projektu.
⋄ testy bezpieczeństwa: zbadanie bezpieczeństwa wdrożonych systemów. Zespół projektowy przyłoży szcze-
gólną uwagę na weryfikację działania wdrażanych firewalli i mechanizmu uwierzytelniania aplikacji webo-
wej. W związku z decyzją o implementacji hostów na systemie operacyjnym Ubuntu 22.04 LTS (Jammy
Jellyfish), w kontekście zbadania bezpieczeństwa hostów (maszyn wirtualnych) zespół będzie bazował na
dokumencie referencyjnym CIS Ubuntu Linux 22.04 LTS Benchmark v1.0.0.
⋄ testy wydajności: zbadanie wydajności systemu informatycznego w szczególności w trakcie np. ataków
brute-force na mechanizm uwierzytelniania aplikacji webowej.
W celu lepszego zobrazowania kolejnych faz projektu wraz z wyróżnionymi zadaniami i czasem realizacji dla
każdej z nich, utworzono wykres realizowanych tematów zadań w funkcji czasu, co przedstawia rysunek 5.
Rysunek 5: Wykres realizowanych tematów zadań w funkcji czasu
9. Szczegółowy schemat projektowanego systemu
Zespół zdecydował się wprowadzić dwupoziomową architekturę sieci (switch, router) w sieci usługowej.
W przypadku chęci rozbudowy sieci (np. dodanie większej ilości urządzeń) lub wprowadzenia segmentacji
(wprowadzenie VLANów), rekomendowane byłoby wprowadzenie trójpoziomowej architektury (access switch,
distribution switch, router). Zmniejszyłoby to obciążenie poszczególnych switchy. Sugerowaną topologię sieci na
schemacie niskopoziomowym przedstawia rysunek 6.
<<obcy język>>
<<obrazek>>
Rysunek 6: Sugerowana topologia sieci: schemat niskopoziomowy
10. Wstępne przygotowanie maszyn wirtualnych dla komponentów serwerowych
<<obcy język>>
Zespół wstępnie przygotował maszyny wirtualne:
• zadecydowano, które komponenty wymagają interfejsu graficznego,
• pobrano i zainstalowano aktualizacje systemowe,
• pobrano niezbędne pakiety związane z implementowanymi usługami,
• przygotowano pliki automatyzujące adresację komponentów (GNS3 usuwa adresacje przy uruchomieniu ma-
szyn wirtualnych).
11. Opis konfiguracji wdrożenia usług na serwerach
11.1. Serwer aplikacyjny
Serwer aplikacyjny został zaimplementowany na maszynie wirtualnej z systemem Ubuntu 22.04 LTS (Jam-
my Jellyfish). Maszyna w wersji Server (brak interfejsu graficznego) została pobrana ze strony https://www.
<<obcy język>>
Serwer aplikacyjny został zaimplementowany przy pomocy framework’u Spring Boot w języku Java, który
powszechnie jest uznawany za platformę o wysokich standardach bezpieczeństwa. Utworzona usługa umożliwia:
• uwierzytelnienie użytkownika, co przedstawia rysunek 7.
Rysunek 7: Mechanizm uwierzytelniania
<<obcy język>>
<<obrazek>>
<<obrazek>>
• po uwierzytelnieniu i autoryzacji, udostępnia stronę główną umożliwiającą poruszanie się po komponentach
aplikacji, co przedstawia rysunek 8.
Rysunek 8: Strona główna
• po kliknięciu w odpowiedni kafelek, użytkownik zostaje przekierowany na podstronę zawierającą bardziej
szczegółowe informacje, co przykładowo przedstawia rysunek 9.
Rysunek 9: Informacje o spółdzielni dostępne po kliknięciu w kafelek ”Spółdzielnia”
Próba dostępu do zasobu bez wcześniejszego uwierzytelnienia (lub w przypadku użytkownika o zbyt niskich
uprawnieniach), skutkuje brakiem dostępu do zasobu oraz przekierowaniem na stronę logowania. Takie dzia-
łanie umożliwia ograniczenie użytkownikom dostępu do poszczególnych, potencjalnie wrażliwych zasobów.
Jest to oparte na modelu kontroli dostępu MAC, w którym poszczególnym użytkownikom nadawane są
uprawnienia dostępu do poszczególnych zasobów.
<<obcy język>>
<<obrazek>>
<<obrazek>>
11.2. Firewall aplikacyjny
Web Application Firewall został zaimplementowany na maszynie wirtualnej z systemem Ubuntu 22.04 LTS
(Jammy Jellyfish). Maszyna w wersji Server (brak interfejsu graficznego) została pobrana z https://www.
<<obcy język>>
Do realizacji WAF zastosowano serwer nginx pełniący rolę serwera proxy z modułem ModSecurity. Pośred-
niczy on w ruchu między klientem (hostem z sieci publicznej) i serwerem aplikacyjnym. Komunikacja na styku
z siecią publiczną jest szyfrowana przy pomocy TLS 1.2. Komunikacja pomiędzy WAF’em i serwerem aplika-
cyjnym nie jest szyfrowana, ponieważ ruch na tym łączu jest w sieci prywatnej i nie może zostać podsłuchany
przez nieuprawnione podmioty.
Zdecydowano się na wprowadzenie następujących reguł:
• logowanie Command Injection w zapytaniu POST (dla zestawu najpopularniejszych poleceń).
• blokowanie ataków siłowych na mechanizm uwierzytelnienia – po 5 błędnych próbach logowania blokowana
jest nazwa użytkownika i źródłowy adres IP (na 10 minut).
Konfiguracja reguł na firewallu aplikacyjnym została przedstawiona na rysunku 10.
Rysunek 10: Reguły utworzone na WAF
12. Opis przygotowania stacji klienckiej
Stacja kliencka została zaimplementowana na maszynie wirtualnej z systemem Ubuntu 22.04 LTS (Jammy
Jellyfish). Maszyna w wersji Desktop (z interfejsem graficznym) została pobrana z https://www.osboxes.org/.
Ze względu na to, że stacja jest jedynie symulacją klienta zewnętrznego, nie wprowadzono na niej dodatkowych
konfiguracji lub usług.
<<obcy język>>
<<obrazek>>
<<obrazek>>
13. Opis konfiguracji wdrożenia infrastruktury sieciowej
Do cron na utworzonych maszyn wirtualnych dodane zostały przygotowane wcześniej pliki automatyzujące
adresację komponentów. Dzięki temu, utworzone maszyny są automatycznie konfigurowane po rozruchu, czyli
są typu ”plug and play”.
Maszyny zaimportowano do emulatora sieci GNS. Dodano również komponenty sieciowe takie jak:
• jeden router Mikrotik w sieci hosta,
• jeden firewall na routerze Mikrotik w sieci hosta,
• jeden router Mikrotik w sieci usługowej,
• jeden firewall na routerze Mikrotik w sieci usługowej,
• jeden unmanaged switch w sieci usługowej (unmanaged ze względu na brak obsługi VLANów).
Pełną topologię utworzonej sieci w emulatorze GNS przedstawia rysunek 11.
Rysunek 11: Topologia utworzonej sieci w GNS3
Wprowadzony firewall hosta przepuszcza jedynie ruch z hosta do sieci Internet i skorelowany ruch z sieci
Internet do hosta. Jest to bezpośrednie zaimplementowanie mechanizmu whitelistowania, gdzie każdy inny rodzaj
ruchu (np. atak Cyber Aktora – nieskorelowany ruch) jest automatycznie odrzucany. Konfigurację routera w sieci
hosta przedstawia rysunek 12. – wyróżniona została konfiguracja firewalla i NAT.
(a) Konfiguracja firewalla – router w sieci hosta
<<obcy język>>
<<obrazek>>
<<obrazek>>
(b) Konfiguracja NAT – router w sieci hosta
Rysunek 12: Konfiguracja routera – sieć hosta
Podobnie, bezpieczeństwo sieci usługowej zostało zapewnione przez firewall na routerze Mikrotik. Zezwala on
jedynie na ruch firewalla aplikacyjnego do Internetu i na skorelowany ruch z Internetu do firewalla aplikacyjnego.
Dzięki temu serwer aplikacyjny (zasób krytyczny) pozostaje chroniony i nie jest bezpośrednio osiągalny z sieci
Internet. Konfigurację routera w sieci usługowej przedstawia rysunek 13. – wyróżniona została konfiguracja
<<obcy język>>
(a) Konfiguracja firewalla – router w sieci usługowej
(b) Konfiguracja NAT i przekierowywania – router w sieci usługowej
Rysunek 13: Konfiguracja routera – sieć usługowa
<<obcy język>>
<<obrazek>>
<<obrazek>>
<<obrazek>>
W celu zbadania systemu przeprowadzono szereg testów funkcjonalnych (spełnienie zakładanych funkcji
w systemie) oraz niefunkcjonalnych (intuicyjność, wydajność, bezpieczeństwo) systemów operacyjnych serwerów
i klienta, usług oraz komunikacji w sieci.
W pierwszej kolejności zbadano funkcjonalność konfiguracji L2 w sieci usługowej. Zweryfikowano, że kompo-
nenty sieci usługowej są wzajemnie osiągalne. Potwierdzenie osiągalności komponentów przy pomocy polecenia
ping przedstawia rysunek 14. Dowodzi to pełnej poprawności funkcjonalności konfiguracji L2 w sieci usługowej.
(a) Test z firewalla aplikacyjnego (b) Test z serwera aplikacyjnego
Rysunek 14: Wzajemna osiągalność komponentów sieci usługowej
Następnie zbadano osiągalność sieci usługowej z Internetu. Potwierdzenie osiągalności sieci usługowej (ro-
utera, firewalla aplikacyjnego i aplikacji webowej) z poziomu hosta przedstawia rysunek 15. Ruch z hosta odwo-
łującego się do publicznego adresu IP routera na portach wystawionych przez usługę aplikacyjną (NAT skon-
figurowany na routerze), zostaje przekierowany do serwera proxy, który następnie pośredniczy w komunikacji
z serwerem aplikacyjnym. Test potwierdza zatem pełną funkcjonalność konfiguracji L3 utworzonej sieci.
Rysunek 15: Osiągalność sieci usługowej z poziomu publicznego hosta
<<obcy język>>
<<obrazek>> <<obrazek>>
<<obrazek>>
14.2. Testy bezpieczeństwa
Zgodnie z założeniami hosty zostały wdrożone na systemie operacyjnym Ubuntu 22.04 LTS (Jammy Jel-
lyfish). W związku z tym, dokumentem referencyjnym w kontekście bezpieczeństwa hostów był CIS Ubuntu
Linux 22.04 LTS Benchmark v1.0.0. Wybór tej wersji systemu operacyjnego był świadomy – Ubuntu udo-
stępnia oprogramowanie usg, które automatycznie doprowadza system do stanu zgodności z benchmarkiem CIS.
Informacje o usgmożna odnaleźć na stronie https://ubuntu.com/security/certifications/docs/usg/cis/
audit. Po wykonaniu audytu wdrożone hosty spełniały wymogi benchmarku na poziomie 97% zgodności. 3% nie-
zgodności odnosi się do rekomendacji w sprawie takich aspektów jak wprowadzenie automatycznego mechanizmu
back-up’ów czy partycjonowania dysku – są one pomijane w przypadku maszyn wirtualnych. Zespół projektowy
doszedł do wniosku, że osiągnięcie 100% zgodności może znacząco wpłynąć na wymagane zasoby wykorzystywa-
ne przez zaprojektowaną architekturę, co przeniosłoby się na zwiększone koszty projektu. Z tego powodu hosty
nie zostały poddane dalszemu utwardzaniu. Wynik utwardzania (ang. hardening) hostów przedstawiony został
na rysunku 16.
Rysunek 16: Wynik audytu badającego zgodność konfiguracji hosta z benchamarkiem CIS Ubuntu Linux 22.04
<<obcy język>>
Zbadano również bezpieczeństwo sieci hosta, które zostało zapewnione przez firewall zaimplementowany na
routerze Mikrotik. Firewall zezwala jedynie na ruch z hosta do sieci Internet i na skorelowany ruch z sieci
Internet do hosta. Jest to zaimplementowany mechanizm whitelistowania.
Podobnie, bezpieczeństwo sieci usługowej zostało zapewnione przez firewall na routerze Mikrotik. Zezwala
on jedynie na ruch firewalla aplikacyjnego do Internetu i na ruch z Internetu na oferowane usługi (dowodzą
tego testy funkcjonalne przedstawione w sekcji 14.1). Test konfiguracji routerów i firewalla poprzez wykonanie
polecenia ping na adresy prywatne komponentów sieci usługowej przedstawia rysunek 17.
Rysunek 17: Próba wykonania polecenia ping na adresy prywatne sieci usługowej
<<obcy język>>
<<obrazek>>
<<obrazek>>
14.3. Testy intuicyjności
Jednym z najważniejszych aspektów do przetestowania było przede wszystkim zbadanie intuicyjności zapro-
jektowanego systemu informatycznego – w szczególności usługi oferowanej przez serwer aplikacyjny (testy ze
stacji klienckiej). Badaną grupą byli współlokatorzy twórców projektu.
W wyniku testów niefunkcjonalnych zespół dostosował wygląd aplikacji webowej do feedbacku, który otrzy-
mał od badanej grupy użytkowników. Strona w ocenie ankietowanych była pierwotnie zbyt statyczna, przez co
nie była atrakcyjna dla odwiedzających. Z tego powodu zespół zdecydował się wprowadzić dynamiczne elementy
takie jak:
• animację najeżdżania na przyciski.
• zmieniające się zdjęcia w dolnej części strony głównej dostępnej po zalogowaniu.
W wyniku wprowadzonych zmian badani użytkownicy potwierdzili, że ich doznania w związku z odwie-
dzaniem strony znacząco się polepszyły. Fragment wprowadzonej animacji (zmiana kolorystyki, ruch strzałki,
rozmiar przycisku) przedstawia rysunek 18. Zmieniające się zdjęcia na stronie głównej przedstawia rysunek 19.
(a) Wygląd przycisku przed najechaniem kursorem (b) Wygląd przycisku po najechaniu kursorem
Rysunek 18: Animacja: zmiana stylu przycisku po najechaniu na niego kursorem
(a) Zdjęcie przed zmianą (przed animacją przesuwania)
<<obcy język>>
<<obrazek>> <<obrazek>>
<<obrazek>>
(b) Zdjęcie po zmianie (po animacji przesuwania)
Rysunek 19: Animacja przesuwania: zmieniające się zdjęcie na stronie głównej
14.4. Testy wydajności
W ramach testów wydajnościowych przeprowadzono ewaluację optymalnych zasobów wymaganych przez
poszczególne komponenty do skutecznego spełnienia narzuconych założeń projektowych. Przeprowadzono testy
systemów operacyjnych i usług stopniowo redukując zasoby przypisane do maszyn wirtualnych. Po wielokrotnych
testach zbadano, że najbardziej optymalną pod względem kosztów i wydajności konfiguracją jest:
• przydzielenie firewallowi aplikacyjnemu pamięci 2GB RAM,
• przydzielenie serwerowi aplikacyjnemu pamięci 2GB RAM,
• przydzielenie hostowi pamięci 4GB RAM.
Przydzielenie mniejszej ilości zasobów skutkuje znaczącym obniżeniem się wydajności poszczególnych kompo-
nentów, a tym samym całego systemu.
Zweryfikowano również wydajność sieci usługowej w trakcie ataku brute-force na mechanizm uwierzytelniania
aplikacji webowej. W związku z zaimplementowaniem reguły na firewallu aplikacyjnym, atak tego typu zostaje
zablokowany po 5 kolejnych próbach. Dzięki temu sieć pozostaje funkcjonalna, a serwer aplikacyjny odpowiada
w oczekiwanym oknie czasowym. Reakcję firewalla aplikacyjnego (zablokowanie dostępu) na zbyt dużą liczbę
błędnych prób logowania przedstawia rysunek 20.
Rysunek 20: Zablokowanie dostępu do strony po zbyt dużej liczbie błędnych prób logowania
<<obcy język>>
<<obrazek>>
<<obrazek>>
W celu wyceny kosztu projektu zespół projektowy rozpisał koszt budowy pojedynczego serwera:
• dysk wewnętrzny: o pojemności min. 50GB. Sugerowanym rozwiązaniem będzie wykorzystanie dysku o po-
jemności min. 250GB, co w przyszłości umożliwi wprowadzenie mechanizmu back-up’ów. Koszt jednego
dysku SSD o pojemności 250GB zaczyna się od 100 zł.
• pamięć RAM: min. 2GB (minimalna pojemność dla DDR4 4GB), koszt około 50 zł.
• płyta główna ze zintegrowaną kartą sieciową: (wymagany socket 1200), koszt ok. 300 zł.
• system operacyjny: dystrybucja Linux Ubuntu 22.04 LTS (Jammy Jellyfish), koszt 0zł.
• obudowa serwerowa: z możliwością zamontowania wewnątrz szafy rackowej, koszt ok. 400 zł
• procesor: ze względu na stosunkowo niskie obciążenie procesora podczas udostępniania usługi serwera www,
rekomendowany jest wybór budżetowej opcji, czyli procesora Intel Core i3-10100F, koszt ok. 300 zł
• zasilanie: o mocy 400W, koszt ok. 100 zł
• chłodzenie: chłodzenie box dołączone do procesora + wentylatory do obudowy, koszt ok. 40 zł
Fizyczny router MikroTik może zrealizować funkcję routera i switcha w jednym urządzeniu. Ze względu na
to, że sieć nie wymaga dużej liczby komponentów, takie rozwiązanie jest jak najbardziej wskazane i preferowane.
Koszt routera MikroTik rozpoczyna się od ok. 300 zł.
Podsumowanie kosztu komponentów sieci usługowej:
• dwa komputery serwerowe: ok. 1290 zł za każdy.
• jeden router MikroTik: ok. 300 zł.
Koszt całkowity zakupu komponentów potrzebnych do realizacji sieci usługowej wynosi zatem ok. 2880 zł.
Jeżeli realizacja sieci hosta byłaby wymagana, to do kosztów należy dodać dodatkowy router MikroTik
i komputer z:
• komponentami jak w komputerze serwerowym z wyjątkiem obudowy,
• obudowa typu mini tower: koszt ok. 100 zł,
• monitor – koszt 400zł
Wówczas całkowity koszt komponentów wzrasta do ok. 4570 zł.
Dodatkowo, w koszty projektu należy włożyć czas jego realizacji przez zespół projektowy. Umowa członków
zespołu projektowego określa ich stawkę godzinową na 30zł/h, co przy czasie realizacji projektu równym 15h,
daje koszt 900 zł.
Podsumowując, koszt pełnej implementacji sieci symulowanej w GNS3 wraz z wliczeniem wynagrodzenia
zespołu projektowego, wyniesie ok. 5470 zł. Koszt poszczególnych elementów w formie graficznej przedstawia
rysunek 21.
Rysunek 21: Kosztorys projektu – koszt poszczególnych elementów
<<obcy język>>
<<obrazek>>
16. Podsumowanie
Celem projektu było zrealizowanie własnego systemu informatycznego, który zostanie uruchomiony w środo-
wisku emulującym sieć teleinformatyczną. Projekt zakładał zrealizowanie niniejszej dokumentacji zawierającej
szczegółowe informacje o procesie projektowania oraz wdrożenia tworzonego systemu informatycznego.
W projekcie wykonano plan stworzenia systemu informatycznego składającego się z:
• komponentu usługowego (sieci usługowej), czyli:
⋄ serwera oferujący usługę aplikacji www,
⋄ firewalla aplikacyjnego,
⋄ routera brzegowego z funkcją firewalla.
• komponentu klienckiego (sieci klienckiej), czyli:
⋄ stacji klienckiej,
⋄ routera brzegowego z funkcją firewalla.
W realizacji projektu wykorzystano następujące technologie / rozwiązania informatyczne:
• VMWare Workstation 17 Player – bezpłatny program, wykorzystany jako środowisko wirtualizacyjne.
• Linux Ubuntu 22.04 LTS (Jammy Jellyfish) – darmowa dystrybucja systemu operacyjnego GNU/Linux
przeznaczona głównie do zastosowań biurowych i domowych.
• GNS3 – bezpłatny emulator sieci teleinformatycznych, wykorzystany do implementacji architektury sieciowej.
• nginx z modułem ModSecurity – serwer www wykorzystany do implementacji firewalla aplikacyjnego w try-
<<obcy język>>
• Spring Boot i język Java – do implementacji serwera aplikacyjnego.
Dodatkowo wykorzystano następujące narzędzia:
• draw.io – do utworzenia schematów.
• Microsoft Visio – do stworzenia wykresu realizowanych tematów zadań w funkcji czasu.
• Microsoft Excel – do stworzenia tabeli ryzyk projektowych.
• LaTeX – do stworzenia niniejszego sprawozdania.
Utworzony projekt został zrealizowany na maszynach wirtualnych w środowisku wirtualnym VMWare Work-
station 17 Player. Na każdej maszynie wirtualnej został zainstalowany system operacyjny Linux Ubuntu 22.04
LTS (Jammy Jellyfish). Następnie maszyny wirtualne zostały dodane do emulatora sieci GNS3, który posłużył
zespołowi do symulacji realnej sieci teleinformatycznej. Do stworzenia pełnej topologii wykorzystano dwa routery
i jeden switch firmy MikroTik. Całość utworzonego systemu została przetestowana pod kątem funkcjonalnym
(zgodność z założeniami) oraz niefunkcjonalnym (bezpieczeństwo, intuicyjność, wydajność). W wyniku testów
zespół projektowy zdecydował się nieznacznie zmienić aplikację webową, aby polepszyć komfort korzystania
z niej przez potencjalnych użytkowników.
Koszt implementacji projektu wynoszący 5470 zł można rozbić na trzy kategorie:
Czasu realizacji projektu to ok. 15h:
W przyszłości projekt można rozbudować o dodatkowe elementy takie jak:
• wprowadzenie mechanizmu automatycznych back-up’ów. Skutkowałoby to zwiększeniem zgodności z bench-
markiem CIS Ubuntu Linux 22.04 LTS Benchmark v1.0.0, ale wymagałoby zwiększonych zasobów.
• wprowadzenie partycjonowania dysków. Skutkowałoby to zwiększeniem zgodności z benchmarkiem CIS
<<obcy język>>
• wprowadzenie segmentacji sieci w celu separacji poszczególnych komponentów w sieci usługowej. Zwiększy-
łoby to bezpieczeństwo poszczególnych komponentów sieciowych po zaimplementowaniu dodatkowych reguł
na firewallu.
<<obcy język>>
"""

In [ ]:
from analysis.nlp_transformations import preprocess_text, split_into_sentences

In [ ]:
preprocessed_text = preprocess_text(og_text)
split = split_into_sentences(preprocessed_text, "pl")

In [ ]:
split